In [2]:
print("Let's Start")

Let's Start


In [ ]:
import pandas as td

df = td.read_csv('/mnt/WindowsD/Projects/commentlens/data/interim/comments.csv' , index_col=0)

df = df.iloc[: , :2]
df.head(4)

,comments,category,word_count,num_stop_words
0,family mormon never tried explain still stare ...,positive,39,13
1,buddhism much lot compatible christianity espe...,positive,196,59
2,seriously say thing first get complex explain ...,negative,86,40
3,learned want teach different focus goal not wr...,neutral,29,15


In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split


def pipeline(
    df,
    models,
    test_size=0.2,
    encoding="embedding",
    max_features=1500,
    embedding_model_name="all-MiniLM-L6-v2",
):
    required_columns = {"comments", "category"}
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise KeyError(f"Missing required columns: {sorted(missing_columns)}")
    if df.empty:
        raise ValueError("Input dataframe is empty")
    if df[["comments", "category"]].isna().any().any():
        raise ValueError("Text and category columns must not contain missing values")
    if not 0 < test_size < 1:
        raise ValueError("test_size must be between 0 and 1")
    if not models:
        raise ValueError("Provide at least one model")
    if not isinstance(encoding, str):
        raise TypeError("encoding must be a string")

    x = df["comments"].astype(str)
    labels = df["category"].astype(str).str.strip().str.lower()
    label_mapping = {"positive": 1, "negative": 2, "neutral": 0}
    y = labels.map(label_mapping)
    if y.isna().any():
        invalid = sorted(labels[y.isna()].unique())
        raise ValueError(f"Unsupported categories: {invalid}")

    encoding = encoding.strip().lower()
    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=test_size,
        random_state=42,
        stratify=y,
    )

    if encoding == "embedding":
        encoder = SentenceTransformer(embedding_model_name)
        x_train_encoded = encoder.encode(x_train.tolist(), show_progress_bar=True)
        x_test_encoded = encoder.encode(x_test.tolist(), show_progress_bar=True)
    elif encoding == "tfidf":
        if max_features < 1:
            raise ValueError("max_features must be positive")

        vectorizer = TfidfVectorizer(
            max_features=max_features,
            dtype=np.float32,
        )
        x_train_encoded = vectorizer.fit_transform(x_train)
        x_test_encoded = vectorizer.transform(x_test)

    elif encoding == "cv":
        if max_features < 1:
            raise ValueError("max_features must be positive")

        vectorizer = CountVectorizer(
            max_features=max_features,
            dtype=np.float32,
        )
        x_train_encoded = vectorizer.fit_transform(x_train)
        x_test_encoded = vectorizer.transform(x_test)

    elif encoding 
    else:
        raise ValueError("encoding must be 'embedding', 'tfidf', or 'cv'")

    results = {}
    metric_labels = sorted(y.unique())
    for model_name, model in models.items():
        model.fit(x_train_encoded, y_train)
        y_pred = model.predict(x_test_encoded)
        results[model_name] = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision_macro": precision_score(
                y_test, y_pred, labels=metric_labels, average="macro", zero_division=0
            ),
            "recall_macro": recall_score(
                y_test, y_pred, labels=metric_labels, average="macro", zero_division=0
            ),
            "f1_macro": f1_score(
                y_test, y_pred, labels=metric_labels, average="macro", zero_division=0
            ),
        }

    return results

In [7]:
import xgboost
import lightgbm
import catboost
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


models = {
    'logisticRegression' : LogisticRegression(),
    'RandomForest' : RandomForestClassifier(),
    'xgboost' : xgboost.XGBClassifier(),
    'lightgbm' : lightgbm.LGBMClassifier(),
    'catboost' : catboost.CatBoostClassifier()
}

In [ ]:
pipeline(df, models, test_size=0.2, encoding="embedding", max_features=1500)

In [ ]:
pipeline(df, models, test_size=0.2, encoding="tfidf", max_features=1500)

In [ ]:
pipeline(df, models, test_size=0.2, encoding="cv", max_features=1500)

In [12]:
d1 = {'logisticRegression': {'accuracy': 0.6948043092867857,
  'precision_macro': 0.6772439109521629,
  'recall_macro': 0.6591857773362365,
  'f1_macro': 0.6629502342058682},
 'RandomForest': {'accuracy': 0.6019364516568935,
  'precision_macro': 0.6388381044097736,
  'recall_macro': 0.5240578205785354,
  'f1_macro': 0.49568115048372346},
 'xgboost': {'accuracy': 0.6708032183281059,
  'precision_macro': 0.6512620073182531,
  'recall_macro': 0.6298726303861962,
  'f1_macro': 0.6326049898304649},
 'lightgbm': {'accuracy': 0.6687576708032184,
  'precision_macro': 0.6512856792568941,
  'recall_macro': 0.6224272289370892,
  'f1_macro': 0.6239814367926154},
 'catboost': {'accuracy': 0.6808945861175508,
  'precision_macro': 0.664174546942914,
  'recall_macro': 0.6398239229343832,
  'f1_macro': 0.642652856275802}}

d2 = {'logisticRegression': {'accuracy': 0.8129005863902905,
  'precision_macro': 0.8096053316178681,
  'recall_macro': 0.7844242723759893,
  'f1_macro': 0.788768221548474},
 'RandomForest': {'accuracy': 0.7809900450020455,
  'precision_macro': 0.7902410284066824,
  'recall_macro': 0.7359161931278004,
  'f1_macro': 0.7431025584912784},
 'xgboost': {'accuracy': 0.7946270285012955,
  'precision_macro': 0.7962566341216261,
  'recall_macro': 0.7657149712591257,
  'f1_macro': 0.7691001526469355},
 'lightgbm': {'accuracy': 0.820673666984863,
  'precision_macro': 0.8183304311799651,
  'recall_macro': 0.7945839461113519,
  'f1_macro': 0.7979924281214377},
 'catboost': {'accuracy': 0.8157643529251329,
  'precision_macro': 0.8172810595186157,
  'recall_macro': 0.7879594404414946,
  'f1_macro': 0.7920843908983445}}

d3 = {'logisticRegression': {'accuracy': 0.8145370244102005,
  'precision_macro': 0.8047956990362759,
  'recall_macro': 0.7918056614257112,
  'f1_macro': 0.7935309513110586},
 'RandomForest': {'accuracy': 0.7758079912723306,
  'precision_macro': 0.7736974953111249,
  'recall_macro': 0.7352602909001494,
  'f1_macro': 0.7400348124354487},
 'xgboost': {'accuracy': 0.7981726442111005,
  'precision_macro': 0.8021038410793683,
  'recall_macro': 0.7705852008327315,
  'f1_macro': 0.7731797813286313},
 'lightgbm': {'accuracy': 0.8257193508795855,
  'precision_macro': 0.8240307726247386,
  'recall_macro': 0.8014250519951917,
  'f1_macro': 0.8043020853475465},
 'catboost': {'accuracy': 0.8142642847402155,
  'precision_macro': 0.8154654835868493,
  'recall_macro': 0.7886255478417269,
  'f1_macro': 0.791756512095235}}


emd = pd.DataFrame(d1)
tf = pd.DataFrame(d2)
cv = pd.DataFrame(d3)

In [29]:
emd.T.sort_values(by='f1_macro' , ascending=False)

,accuracy,precision_macro,recall_macro,f1_macro
logisticRegression,0.694804,0.677244,0.659186,0.662950
catboost,0.680895,0.664175,0.639824,0.642653
xgboost,0.670803,0.651262,0.629873,0.632605
lightgbm,0.668758,0.651286,0.622427,0.623981
RandomForest,0.601936,0.638838,0.524058,0.495681


In [30]:
tf.T.sort_values(by='f1_macro' , ascending=False)

,accuracy,precision_macro,recall_macro,f1_macro
lightgbm,0.820674,0.818330,0.794584,0.797992
catboost,0.815764,0.817281,0.787959,0.792084
logisticRegression,0.812901,0.809605,0.784424,0.788768
xgboost,0.794627,0.796257,0.765715,0.769100
RandomForest,0.780990,0.790241,0.735916,0.743103


In [31]:
cv.T.sort_values(by='f1_macro' , ascending=False)

,accuracy,precision_macro,recall_macro,f1_macro
lightgbm,0.825719,0.824031,0.801425,0.804302
logisticRegression,0.814537,0.804796,0.791806,0.793531
catboost,0.814264,0.815465,0.788626,0.791757
xgboost,0.798173,0.802104,0.770585,0.773180
RandomForest,0.775808,0.773697,0.735260,0.740035


In [9]:
x = df["comments"].astype(str)
labels = df["category"].astype(str).str.strip().str.lower()
label_mapping = {"positive": 1, "negative": 2, "neutral": 0}
y = labels.map(label_mapping)
if y.isna().any():
    invalid = sorted(labels[y.isna()].unique())
    raise ValueError(f"Unsupported categories: {invalid}")

x_train_, x_test_, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )


In [49]:
metric_labels = sorted(y.unique())
max_features_values = (500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000)
lgmbm_scores_cv = {}

for feature in max_features_values:
    vectorizer = CountVectorizer(max_features=feature, dtype=np.float32)
    x_train_trans = vectorizer.fit_transform(x_train_)
    x_test_trans = vectorizer.transform(x_test_)

    model = lightgbm.LGBMClassifier()
    model.fit(x_train_trans, y_train)
    y_pred = model.predict(x_test_trans)

    lgmbm_scores_cv[f"{feature}_features"] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(
            y_test, y_pred, labels=metric_labels,
            average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            y_test, y_pred, labels=metric_labels,
            average="macro", zero_division=0
        ),
        "f1_macro": f1_score(
            y_test, y_pred, labels=metric_labels,
            average="macro", zero_division=0
        ),
    }

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015286 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2883
[LightGBM] [Info] Number of data points in the train set: 29329, number of used features: 500
[LightGBM] [Info] Start training from score -1.064557
[LightGBM] [Info] Start training from score -0.843611
[LightGBM] [Info] Start training from score -1.491810
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.032411 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5005
[LightGBM] [Info] Number of data points in the train set: 29329, number of used features: 999
[LightGBM] [Info] Start training from score -1.064557
[LightGBM] [Info] Start training from score -0.843611
[LightGBM] [Info] Star

In [51]:
lg_df = pd.DataFrame(lgmbm_scores_cv).T
lg_df.sort_values(by='f1_macro' , ascending=False)

,accuracy,precision_macro,recall_macro,f1_macro
5000_features,0.846993,0.846713,0.824146,0.828335
4500_features,0.846993,0.846713,0.824146,0.828335
4000_features,0.846993,0.846713,0.824146,0.828335
3000_features,0.846720,0.846411,0.824005,0.828149
2500_features,0.845357,0.844058,0.822755,0.826522
3500_features,0.845766,0.845188,0.822231,0.826445
2000_features,0.841811,0.840438,0.818099,0.821963
1500_features,0.825719,0.824031,0.801425,0.804302
1000_features,0.801446,0.800560,0.775715,0.777578
500_features,0.740488,0.743997,0.709291,0.707780


In [ ]:
metric_labels = sorted(y.unique())
max_features_values = (500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000)
lgmbm_scores_tf = {}

for feature in max_features_values:
    vectorizer = CountVectorizer(max_features=feature, dtype=np.float32)
    x_train_trans = vectorizer.fit_transform(x_train_)
    x_test_trans = vectorizer.transform(x_test_)

    model = lightgbm.LGBMClassifier()
    model.fit(x_train_trans, y_train)
    y_pred = model.predict(x_test_trans)

    lgmbm_scores_tf[f"{feature}_features"] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(
            y_test, y_pred, labels=metric_labels,
            average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            y_test, y_pred, labels=metric_labels,
            average="macro", zero_division=0
        ),
        "f1_macro": f1_score(
            y_test, y_pred, labels=metric_labels,
            average="macro", zero_division=0
        ),
    }

In [16]:
lg_df = pd.DataFrame(lgmbm_scores_tf).T
lg_df.sort_values(by='f1_macro' , ascending=False)

,accuracy,precision_macro,recall_macro,f1_macro
5000_features,0.846993,0.846713,0.824146,0.828335
4500_features,0.846993,0.846713,0.824146,0.828335
4000_features,0.846993,0.846713,0.824146,0.828335
3000_features,0.846720,0.846411,0.824005,0.828149
2500_features,0.845357,0.844058,0.822755,0.826522
3500_features,0.845766,0.845188,0.822231,0.826445
2000_features,0.841811,0.840438,0.818099,0.821963
1500_features,0.825719,0.824031,0.801425,0.804302
1000_features,0.801446,0.800560,0.775715,0.777578
500_features,0.740488,0.743997,0.709291,0.707780


In [ ]:
vectorizer = TfidfVectorizer(max_features=3000, dtype=np.float32)
x_train_trans = vectorizer.fit_transform(x_train_)
x_test_trans = vectorizer.transform(x_test_)

model = lightgbm.LGBMClassifier()
model.fit(x_train_trans, y_train)
y_pred = model.predict(x_test_trans)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.281644 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 114319
[LightGBM] [Info] Number of data points in the train set: 29329, number of used features: 2947
[LightGBM] [Info] Start training from score -1.064557
[LightGBM] [Info] Start training from score -0.843611
[LightGBM] [Info] Start training from score -1.491810


In [ ]:
from sklearn.metrics import classification_report
# cv at max_features 3000
print(classification_report(y_test , y_pred))

              precision    recall  f1-score   support

           0       0.81      0.98      0.88      2529
           1       0.89      0.85      0.87      3154
           2       0.84      0.65      0.73      1650

    accuracy                           0.85      7333
   macro avg       0.85      0.82      0.83      7333
weighted avg       0.85      0.85      0.84      7333



In [ ]:
from sklearn.metrics import classification_report
# tfidf at max_features 3000

print(classification_report(y_test , y_pred))

              precision    recall  f1-score   support

           0       0.81      0.97      0.88      2529
           1       0.89      0.85      0.87      3154
           2       0.84      0.66      0.74      1650

    accuracy                           0.85      7333
   macro avg       0.85      0.83      0.83      7333
weighted avg       0.85      0.85      0.84      7333



In [32]:
from lightgbm import LGBMClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix

vectorizer = TfidfVectorizer(
    max_features=6000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    dtype=np.float32,
)

x_train_tf = vectorizer.fit_transform(x_train_)
x_test_tf = vectorizer.transform(x_test_)

model = LGBMClassifier(
    class_weight="balanced",
    n_estimators=400,
    learning_rate=0.08,
    num_leaves=31,
    colsample_bytree=0.8,
    random_state=42,
)

model.fit(x_train_tf, y_train)
y_pred = model.predict(x_test_tf)

print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.242563 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 130133
[LightGBM] [Info] Number of data points in the train set: 29329, number of used features: 4344
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
              precision    recall  f1-score   support

           0       0.86      0.96      0.91      2529
           1       0.91      0.85      0.88      3154
           2       0.81      0.77      0.79      1650

    accuracy                           0.87      7333
   macro avg       0.86      0.86      0.86      7333
weighted avg       0.87      0.87      0.87      7333

Confusion Matrix:
 [[2427   40   62]
 [ 241 2673  240]
 [ 165  216 1269]]


In [38]:
from lightgbm import LGBMClassifier
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.metrics import classification_report, confusion_matrix

vectorizer = CountVectorizer(
    max_features=6000,
    ngram_range=(1, 2),
    dtype=np.float32,
)

x_train_tf = vectorizer.fit_transform(x_train_)
x_test_tf = vectorizer.transform(x_test_)

model = LGBMClassifier(
    class_weight="balanced",
    n_estimators=400,
    learning_rate=0.08,
    num_leaves=31,
    colsample_bytree=0.8,
    random_state=42,
)

model.fit(x_train_tf, y_train)
y_pred = model.predict(x_test_tf)

print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.738909 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15402
[LightGBM] [Info] Number of data points in the train set: 29329, number of used features: 4344
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
              precision    recall  f1-score   support

           0       0.85      0.97      0.91      2529
           1       0.92      0.85      0.88      3154
           2       0.82      0.78      0.80      1650

    accuracy                           0.87      7333
   macro avg       0.86      0.86      0.86      7333
weighted avg       0.87      0.87      0.87      7333

Confusion Matrix:
 [[2449   33   47]
 [ 249 2672  233]
 [ 170  199 1281]]


In [ ]:
from lightgbm import LGBMClassifier
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

vectorizer = CountVectorizer(
    max_features=6000,
    ngram_range=(1, 2),
    dtype=np.float32,
)

x_train_tf = vectorizer.fit_transform(x_train_)
x_test_tf = vectorizer.transform(x_test_)

model = LGBMClassifier(
    class_weight="balanced",
    n_estimators=400,
    learning_rate=0.08,
    num_leaves=31,
    colsample_bytree=0.8,
    random_state=42,
)

model.fit(x_train_tf, y_train)
y_pred = model.predict(x_test_tf)

print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [40]:
model.score(x_test_tf , y_test)

0.8730396836219828

In [41]:
model.score(x_train_tf , y_train)

0.9242047120597361

In [42]:
from sklearn.model_selection import learning_curve

lr = learning_curve(model , x_train_tf , y_train )

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.272432 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1193
[LightGBM] [Info] Number of data points in the train set: 2346, number of used features: 335
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spl

KeyboardInterrupt: 